In [1]:
import os
import pandas as pd
import numpy as np
from openpyxl import load_workbook
 
DATA_DIR = '../data/raw/'
 
# ---- Legacy file (2020-2023) ------------------------------------------------
LEGACY_FILE = '2020-2023.xlsx'
LEGACY_RAW_TABS = ['2020原始', '2021原始', '2022原始', '2023原始']
LEGACY_COA_TABS = ['COA4', 'COA5', 'COA6']
 
# ---- Modern IVL files -------------------------------------------------------
IVL_FILES = [
    '2024IVL夏季赛常规赛.xlsx',
    '2024IVL夏季赛季后赛.xlsx',
    '2024IVL秋季赛常规赛.xlsx',
    '2024IVL秋季赛季后赛.xlsx',
    '2025IVL夏季赛常规赛.xlsx',
    '2025IVL夏季赛季后赛.xlsx',
    '2025IVL秋季赛常规赛.xlsx',
    '2025IVL秋季赛季后赛.xlsx',
]
 
# ---- Modern IJL files -------------------------------------------------------
IJL_FILES = [
    '2024IJL夏季赛常规赛.xlsx',
    '2024IJL秋季赛季后赛.xlsx',
    '2025IJL夏季赛常规赛.xlsx',
    '2025IJL夏季赛季后赛.xlsx',
    '2025IJL秋季赛常规赛.xlsx',
    '2025IJL秋季赛季后赛.xlsx',
]
 
# ---- COA main event files (excluding Japan qualifiers) ----------------------
COA_FILES = [
    'COA8 全球总决赛小组赛.xlsx',
    'COA8 全球总决赛淘汰赛.xlsx',
    'COA9 全球总决赛小组赛.xlsx',
    'COA9 全球总决赛淘汰赛.xlsx',
]
 
# ---- Special / excluded -----------------------------------------------------
EXCLUDED_FILES = [
    '2025IVS.xlsx',                    # only one year, small sample
    'COA8 日本赛区预选赛.xlsx',          # regional qualifier
    'COA9 日本赛区预选赛.xlsx',          # regional qualifier
]
 
ALL_MODERN_FILES = IVL_FILES + IJL_FILES + COA_FILES
MODERN_RAW_SHEET    = '原始数据'
MODERN_PLAYER_SHEET = '赛后数据'
MODERN_GAME_SHEET   = '对局数据'
 
print("File registry loaded.")
print(f"  Legacy tabs (raw):  {LEGACY_RAW_TABS}")
print(f"  Legacy tabs (COA):  {LEGACY_COA_TABS}")
print(f"  Modern IVL files:   {len(IVL_FILES)}")
print(f"  Modern IJL files:   {len(IJL_FILES)}")
print(f"  COA main files:     {len(COA_FILES)}")
print(f"  Excluded files:     {len(EXCLUDED_FILES)}")
 
 

File registry loaded.
  Legacy tabs (raw):  ['2020原始', '2021原始', '2022原始', '2023原始']
  Legacy tabs (COA):  ['COA4', 'COA5', 'COA6']
  Modern IVL files:   8
  Modern IJL files:   6
  COA main files:     4
  Excluded files:     3


In [2]:
print("=== File existence check ===\n")
 
all_files_to_check = (
    [LEGACY_FILE] +
    ALL_MODERN_FILES +
    EXCLUDED_FILES
)
 
missing = []
for f in all_files_to_check:
    path = os.path.join(DATA_DIR, f)
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / 1e6 if exists else 0
    status = f"OK  ({size_mb:.1f} MB)" if exists else "MISSING"
    print(f"  [{status}]  {f}")
    if not exists:
        missing.append(f)
 
if missing:
    print(f"\n⚠️  {len(missing)} file(s) missing — fix paths before continuing")
else:
    print(f"\n✅ All files found")
 
 

=== File existence check ===

  [OK  (3.1 MB)]  2020-2023.xlsx
  [OK  (1.6 MB)]  2024IVL夏季赛常规赛.xlsx
  [OK  (1.4 MB)]  2024IVL夏季赛季后赛.xlsx
  [OK  (1.8 MB)]  2024IVL秋季赛常规赛.xlsx
  [OK  (1.5 MB)]  2024IVL秋季赛季后赛.xlsx
  [OK  (2.0 MB)]  2025IVL夏季赛常规赛.xlsx
  [OK  (1.7 MB)]  2025IVL夏季赛季后赛.xlsx
  [OK  (2.0 MB)]  2025IVL秋季赛常规赛.xlsx
  [OK  (1.8 MB)]  2025IVL秋季赛季后赛.xlsx
  [OK  (1.5 MB)]  2024IJL夏季赛常规赛.xlsx
  [OK  (1.5 MB)]  2024IJL秋季赛季后赛.xlsx
  [OK  (1.9 MB)]  2025IJL夏季赛常规赛.xlsx
  [OK  (1.7 MB)]  2025IJL夏季赛季后赛.xlsx
  [OK  (1.9 MB)]  2025IJL秋季赛常规赛.xlsx
  [OK  (1.7 MB)]  2025IJL秋季赛季后赛.xlsx
  [OK  (1.8 MB)]  COA8 全球总决赛小组赛.xlsx
  [OK  (1.7 MB)]  COA8 全球总决赛淘汰赛.xlsx
  [OK  (2.2 MB)]  COA9 全球总决赛小组赛.xlsx
  [OK  (1.8 MB)]  COA9 全球总决赛淘汰赛.xlsx
  [OK  (1.7 MB)]  2025IVS.xlsx
  [OK  (1.6 MB)]  COA8 日本赛区预选赛.xlsx
  [OK  (1.8 MB)]  COA9 日本赛区预选赛.xlsx

✅ All files found


In [28]:
print("=== Sheet names in modern files ===\n")
 
REQUIRED_SHEETS = {MODERN_RAW_SHEET, MODERN_PLAYER_SHEET}
problems = []
 
for f in ALL_MODERN_FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        continue
    wb = load_workbook(path, read_only=True)
    sheets = set(wb.sheetnames)
    has_raw    = MODERN_RAW_SHEET    in sheets
    has_player = MODERN_PLAYER_SHEET in sheets
    has_game   = MODERN_GAME_SHEET   in sheets
    status = "✅" if (has_raw and has_player) else "⚠️ "
    print(f"  {status} {f}")
    print(f"       原始数据: {has_raw}   赛后数据: {has_player}   对局数据: {has_game}")
    if not (has_raw and has_player):
        problems.append(f)
 
if problems:
    print(f"\n⚠️  {len(problems)} file(s) missing expected sheets: {problems}")
else:
    print(f"\n✅ All modern files have required sheets")

# =============================================================================
# Canonical column names — what every cleaned dataframe will use downstream
# =============================================================================

# Maps any variant of a column name to its canonical English form.
# Add new aliases here as you discover them in new files.
COLUMN_ALIASES = {
    # --- Game-level metadata ---
    '本页出错':          'page_error',
    '阶段':              'stage',
    '日期':              'date',
    '月':                'month',
    '日':                'day',
    '时间':              'time',
    '场次':              'match_num',
    '半场':              'half',
    '主场':              'home_team',
    '客场':              'away_team',
    
    # --- Hunter side ---
    '屠队主客':          'hunter_side',
    '屠队':              'hunter_team',
    '屠名':              'hunter_player',
    '屠选':              'hunter_character',
    '屠ID':              'hunter_player',     # modern alias for 屠名
    
    # --- Survivor side ---
    '人队':              'survivor_team',
    
    # --- Outcome / scoring ---
    '胜利方':            'winner_side',
    '局分':              'half_score',
    '局总分':            'game_score',
    '场分':              'match_score',
    '主分':              'home_score',
    '客分':              'away_score',
    '剩机台数':          'gens_remaining',
    '剩机':              'gens_remaining',    # modern alias
    
    # --- Map / draft ---
    '地图':              'map_name',
    'BAN图':             'banned_map',
    'BAN图.1':           'banned_map_2',
    'BAN图1':            'banned_map',
    'BAN图2':            'banned_map_2',
    '地图主客':          'map_picker_side',
    '地图选择方主客':    'map_picker_side',   # 2020 form
    '选图队伍':          'map_picker_team',
    '地图选择方队名':    'map_picker_team',   # 2021 form
    '选边队伍':          'side_picker_team',
    
    # --- Match meta ---
    '用时 (分)':         'duration_min',
    '用时 (秒)':         'duration_sec',
    'MVP':               'mvp',
    '暂停或重赛':        'paused_or_rematch',
    '备注':              'notes',
}


 

=== Sheet names in modern files ===

  ✅ 2024IVL夏季赛常规赛.xlsx
       原始数据: True   赛后数据: True   对局数据: False
  ✅ 2024IVL夏季赛季后赛.xlsx
       原始数据: True   赛后数据: True   对局数据: False
  ✅ 2024IVL秋季赛常规赛.xlsx
       原始数据: True   赛后数据: True   对局数据: True
  ✅ 2024IVL秋季赛季后赛.xlsx
       原始数据: True   赛后数据: True   对局数据: False
  ✅ 2025IVL夏季赛常规赛.xlsx
       原始数据: True   赛后数据: True   对局数据: True
  ✅ 2025IVL夏季赛季后赛.xlsx
       原始数据: True   赛后数据: True   对局数据: True
  ✅ 2025IVL秋季赛常规赛.xlsx
       原始数据: True   赛后数据: True   对局数据: True
  ✅ 2025IVL秋季赛季后赛.xlsx
       原始数据: True   赛后数据: True   对局数据: True
  ✅ 2024IJL夏季赛常规赛.xlsx
       原始数据: True   赛后数据: True   对局数据: False
  ✅ 2024IJL秋季赛季后赛.xlsx
       原始数据: True   赛后数据: True   对局数据: False
  ✅ 2025IJL夏季赛常规赛.xlsx
       原始数据: True   赛后数据: True   对局数据: True
  ✅ 2025IJL夏季赛季后赛.xlsx
       原始数据: True   赛后数据: True   对局数据: True
  ✅ 2025IJL秋季赛常规赛.xlsx
       原始数据: True   赛后数据: True   对局数据: True
  ✅ 2025IJL秋季赛季后赛.xlsx
       原始数据: True   赛后数据: True   对局数据: True
  ✅ COA8 全球总决赛小组赛.xl

In [43]:
# =============================================================================
# Canonical column names — what every cleaned dataframe will use downstream
# =============================================================================

# Maps any variant of a column name to its canonical English form.
# Add new aliases here as you discover them in new files.
ADDITIONAL_ALIASES = {
    # --- Hunter-side game stats (modern 赛后数据) ---
    # These are aggregate hunter performance per game half
    '击倒':              'hunter_knockdowns',
    '命中':              'hunter_hits',
    '震慑':              'hunter_stuns',
    '破板':              'hunter_boards_broken',
    '角色.4':            'hunter_character',     # hunter's character in 赛后数据
    '角色.5':            'hunter_talent',        # hunter's talent/skill choice
    
    # --- Match-level outcome aggregates ---
    '总逃脱':            'total_escapes',
    '逃生数':            'total_escapes',         # variant
    '淘汰':              'eliminations',
    '淘汰数':            'eliminations',
    '总破译进度':        'total_repair_progress',
    '总牵制时长':        'total_harassment',
    '游戏用时':          'game_duration_sec',
    '胜利':              'winner_side',           # alternative form
    
    # --- Draft (older 2020 form, before phased drafts existed) ---
    '人BAN':             'survivor_ban_2020',
    '人选':              'survivor_pick_2020',
    '屠BAN':             'hunter_ban_2020',
    
    # --- Auto-generated ban/pick aggregates (2023+) ---
    '求生者首抢禁用【自动生成】':  'survivor_first_ban_auto',
    '求生者全局禁用【自动生成】':  'survivor_global_ban_auto',
    '监管者全局禁用【自动生成】':  'hunter_global_ban_auto',
    
    # --- Talent / supplementary picks ---
    '辅助特质':          'support_talent',
    '辅助特质1':         'support_talent',        # 2021-2022 variant
    '底牌后':            'post_trump_talent',
    
    # --- Misc legacy fields ---
    '光明之星':          'mvp',         # 2020-only MVP-like award
    '场次.1':            'match_num_2',           # legacy duplicate
}

COLUMN_ALIASES.update(ADDITIONAL_ALIASES)

# --- Survivor slot columns (1-4) — built programmatically since they're indexed ---
SURVIVOR_ALIASES = {}
for i in range(1, 5):
    suffix = '' if i == 1 else f'.{i-1}'
    # ID columns
    SURVIVOR_ALIASES[f'求生者{i}ID'] = f'survivor{i}_player'
    SURVIVOR_ALIASES[f'人ID{i}']     = f'survivor{i}_player'
    # Character columns
    SURVIVOR_ALIASES[f'使用角色{suffix}'] = f'survivor{i}_character'
    SURVIVOR_ALIASES[f'角色{suffix}']     = f'survivor{i}_character'
    # Stat columns (pandas appends .1, .2, .3 for duplicate names)
    SURVIVOR_ALIASES[f'修机进度{suffix}'] = f'survivor{i}_repairs'
    SURVIVOR_ALIASES[f'修机{suffix}']     = f'survivor{i}_repairs'
    SURVIVOR_ALIASES[f'救人数{suffix}']   = f'survivor{i}_rescues'
    SURVIVOR_ALIASES[f'救人{suffix}']     = f'survivor{i}_rescues'
    SURVIVOR_ALIASES[f'治疗数{suffix}']   = f'survivor{i}_heals'
    SURVIVOR_ALIASES[f'治疗{suffix}']     = f'survivor{i}_heals'
    SURVIVOR_ALIASES[f'砸板命中{suffix}'] = f'survivor{i}_boards'
    SURVIVOR_ALIASES[f'砸板{suffix}']     = f'survivor{i}_boards'
    SURVIVOR_ALIASES[f'牵制时长{suffix}'] = f'survivor{i}_harassment'
    SURVIVOR_ALIASES[f'牵制{suffix}']     = f'survivor{i}_harassment'
    SURVIVOR_ALIASES[f'结果{suffix}']     = f'survivor{i}_result'

COLUMN_ALIASES.update(SURVIVOR_ALIASES)

# --- Draft phase columns (consistent across years) ---
DRAFT_ALIASES = {
    '第一阶段人BAN':  'phase1_survivor_ban',
    '第一阶段屠BAN':  'phase1_hunter_ban',
    '第一阶段人PICK': 'phase1_survivor_pick',
    '第二阶段人BAN':  'phase2_survivor_ban',
    '第二阶段人PICK': 'phase2_survivor_pick',
    '第三阶段人BAN':  'phase3_survivor_ban',
    '第三阶段人PICK': 'phase3_survivor_pick',
}
COLUMN_ALIASES.update(DRAFT_ALIASES)


def normalize_columns(df, alias_map=COLUMN_ALIASES, verbose=False):
    """
    Rename columns using the alias map.
    Columns not in the map are left as-is.
    Returns (renamed_df, list_of_unmapped_cols).
    """
    unmapped = []
    new_names = {}
    for col in df.columns:
        col_str = str(col).strip()
        if col_str in alias_map:
            new_names[col] = alias_map[col_str]
        elif col_str.startswith('Unnamed') or col_str.startswith('_'):
            pass  # leave system columns alone
        else:
            unmapped.append(col_str)
    
    renamed = df.rename(columns=new_names)
    
    if verbose and unmapped:
        print(f"Unmapped columns: {unmapped}")
    
    return renamed, unmapped
df = read_modern_sheet('2024IVL夏季赛季后赛.xlsx', '原始数据')
df, unmapped = normalize_columns(df, verbose=True)
print(df.columns.tolist()[:20])
print(f"\nUnmapped count: {len(unmapped)}")

Unmapped columns: ['page_error', 'month', 'day', 'time', 'home_team', 'away_team', 'match_num', 'half', 'hunter_side', 'survivor_team', 'hunter_team', 'hunter_player', 'half_score', 'game_score', 'match_score', 'home_score', 'away_score', 'winner_side', 'banned_map', 'map_name', 'survivor_first_ban_auto', 'phase1_survivor_ban', 'phase1_hunter_ban', 'phase1_survivor_pick', 'phase2_survivor_ban', 'phase2_survivor_pick', 'phase3_survivor_ban', 'phase3_survivor_pick', 'hunter_character', 'map_picker_side', 'map_picker_team', 'side_picker_team']
['page_error', 'month', 'day', 'time', 'home_team', 'away_team', 'match_num', 'half', 'hunter_side', 'survivor_team', 'hunter_team', 'hunter_player', 'half_score', 'Unnamed: 13', 'game_score', 'match_score', 'Unnamed: 16', 'Unnamed: 17', 'home_score', 'away_score']

Unmapped count: 32


In [72]:
# Known marker columns that should appear in correct header rows
MODERN_RAW_MARKERS    = {'本页出错', '主场', '客场', '胜利方', '屠选'}
MODERN_PLAYER_MARKERS = {'人ID1', '屠ID', '修机', '牵制'}
LEGACY_MARKERS        = {'阶段', '主场', '客场', '屠队', '地图'}

def detect_header_row(path, sheet_name, markers, max_search=5):
    """
    Try header rows 0 through max_search-1.
    Return the row that contains the most marker columns.
    """
    best_row, best_score = 0, -1
    for h in range(max_search):
        try:
            df = pd.read_excel(path, sheet_name=sheet_name, header=h, nrows=0)
            cols = {str(c).strip() for c in df.columns}
            score = len(cols & markers)
            if score > best_score:
                best_row, best_score = h, score
        except Exception:
            continue
    return best_row, best_score

def read_legacy_tab(tab_name, nrows=None, normalize=True):
    """Read legacy tab, auto-detect header, optionally normalize columns."""
    path = os.path.join(DATA_DIR, LEGACY_FILE)
    header_row, score = detect_header_row(path, tab_name, LEGACY_MARKERS)
    df = pd.read_excel(path, sheet_name=tab_name, header=header_row, nrows=nrows)
    df = df.dropna(how='all')
    df['_source'] = f'legacy:{tab_name}'
    df['_header_row'] = header_row
    if normalize:
        df, _ = normalize_columns(df)
    if 'hunter_team' in df.columns:
        df = df[df['hunter_team'].notna()].copy()
    return df

def read_modern_sheet(filename, sheet_name, nrows=None, normalize=True):
    """Read modern sheet, auto-detect header, optionally normalize columns."""
    path = os.path.join(DATA_DIR, filename)
    markers = (MODERN_RAW_MARKERS if sheet_name == MODERN_RAW_SHEET 
               else MODERN_PLAYER_MARKERS)
    header_row, score = detect_header_row(path, sheet_name, markers)
    if score < 2:
        print(f"⚠️  Low confidence: {filename}:{sheet_name} (score={score})")
    df = pd.read_excel(path, sheet_name=sheet_name, header=header_row, nrows=nrows)
    df = df.dropna(how='all')
    df['_source'] = f'{filename}:{sheet_name}'
    df['_header_row'] = header_row
    if normalize:
        df, _ = normalize_columns(df)
    if 'hunter_team' in df.columns:
        df = df[df['hunter_team'].notna()].copy()
    return df

In [73]:
print("=== Legacy raw tab schemas (English names) ===\n")

legacy_schemas = {}
for tab in LEGACY_RAW_TABS:
    df = read_legacy_tab(tab, nrows=0)
    named_cols = [c for c in df.columns 
                  if not str(c).startswith('Unnamed') 
                  and not str(c).startswith('_')]
    legacy_schemas[tab] = set(named_cols)
    print(f"{tab}  ({len(named_cols)} mapped cols):")
    print(f"  {sorted(named_cols)}\n")

print("--- Columns added vs previous year ---")
tabs = LEGACY_RAW_TABS
for i in range(1, len(tabs)):
    prev, curr = tabs[i-1], tabs[i]
    added   = legacy_schemas[curr] - legacy_schemas[prev]
    removed = legacy_schemas[prev] - legacy_schemas[curr]
    print(f"\n{prev} → {curr}")
    if added:   print(f"  ADDED:   {sorted(added)}")
    if removed: print(f"  REMOVED: {sorted(removed)}")

=== Legacy raw tab schemas (English names) ===

2020原始  (40 mapped cols):
  ['away_score', 'away_team', 'date', 'game_score', 'gens_remaining', 'half_score', 'home_score', 'home_team', 'hunter_ban_2020', 'hunter_character', 'hunter_player', 'hunter_side', 'hunter_team', 'map_name', 'map_picker_side', 'match_num', 'match_num_2', 'match_score', 'notes', 'stage', 'star_of_light', 'survivor1_character', 'survivor1_harassment', 'survivor1_player', 'survivor1_repairs', 'survivor2_character', 'survivor2_harassment', 'survivor2_player', 'survivor2_repairs', 'survivor3_character', 'survivor3_harassment', 'survivor3_player', 'survivor3_repairs', 'survivor4_character', 'survivor4_harassment', 'survivor4_player', 'survivor4_repairs', 'survivor_ban_2020', 'survivor_pick_2020', 'winner_side']

2021原始  (58 mapped cols):
  ['(底牌后)     沒带填0', 'away_score', 'away_team', 'banned_map', 'date', 'duration_min', 'duration_sec', 'game_score', 'gens_remaining', 'half_score', 'home_score', 'home_team', 'hunter_

In [65]:
print("=== Legacy raw tab schemas ===\n")
 
legacy_schemas = {}
for tab in LEGACY_RAW_TABS:
    df = read_legacy_tab(tab, nrows=0)
    named_cols = clean_col_names(df.columns)
    legacy_schemas[tab] = set(named_cols)
    print(f"{tab}  ({len(df.columns)} total cols, {len(named_cols)} named):")
    print(f"  {named_cols}")
    print()
 
# What columns were added each year?
print("--- Columns added vs previous year ---")
tabs = LEGACY_RAW_TABS
for i in range(1, len(tabs)):
    prev, curr = tabs[i-1], tabs[i]
    added   = legacy_schemas[curr] - legacy_schemas[prev]
    removed = legacy_schemas[prev] - legacy_schemas[curr]
    print(f"\n{prev} → {curr}")
    if added:
        print(f"  ADDED:   {sorted(added)}")
    if removed:
        print(f"  REMOVED: {sorted(removed)}")
    if not added and not removed:
        print(f"  (no named column changes)")

=== Legacy raw tab schemas ===

2020原始  (53 total cols, 42 named):
  ['stage', 'date', 'match_num', 'home_team', 'away_team', 'match_num_2', 'hunter_side', 'hunter_team', 'hunter_player', 'half_score', 'game_score', 'match_score', 'home_score', 'away_score', 'winner_side', 'map_name', 'survivor_ban_2020', 'hunter_ban_2020', 'survivor_pick_2020', 'hunter_character', 'map_picker_side', 'gens_remaining', 'survivor1_player', 'survivor1_character', 'survivor1_harassment', 'survivor1_repairs', 'survivor2_player', 'survivor2_character', 'survivor2_harassment', 'survivor2_repairs', 'survivor3_player', 'survivor3_character', 'survivor3_harassment', 'survivor3_repairs', 'survivor4_player', 'survivor4_character', 'survivor4_harassment', 'survivor4_repairs', 'star_of_light', 'notes', '_source', '_header_row']

2021原始  (70 total cols, 60 named):
  ['stage', 'date', 'match_num', 'home_team', 'away_team', 'match_num_2', 'hunter_side', 'survivor_team', 'hunter_team', 'hunter_player', 'half_score', 'ga

In [66]:
print("=== Unmapped columns audit ===\n")

all_unmapped = set()

# Legacy
for tab in LEGACY_RAW_TABS + LEGACY_COA_TABS:
    df = read_legacy_tab(tab, nrows=0, normalize=False)
    _, unmapped = normalize_columns(df)
    if unmapped:
        print(f"legacy:{tab}: {unmapped}")
        all_unmapped.update(unmapped)

# Modern
for f in ALL_MODERN_FILES:
    if not os.path.exists(os.path.join(DATA_DIR, f)):
        continue
    for sheet in [MODERN_RAW_SHEET, MODERN_PLAYER_SHEET]:
        df = read_modern_sheet(f, sheet, nrows=0, normalize=False)
        _, unmapped = normalize_columns(df)
        if unmapped:
            print(f"{f}:{sheet}: {unmapped}")
            all_unmapped.update(unmapped)

print(f"\n=== Total unique unmapped columns: {len(all_unmapped)} ===")
for c in sorted(all_unmapped):
    print(f"  {c}")

=== Unmapped columns audit ===

legacy:2021原始: ['(底牌后)     沒带填0']
legacy:2022原始: ['(底牌后)     没带写0']
legacy:COA5: ['(底牌后)   没带写0']
⚠️  Low confidence: 2024IVL夏季赛常规赛.xlsx:赛后数据 (score=0)
2024IVL夏季赛常规赛.xlsx:赛后数据: ['赛后数据', '检查员用']
⚠️  Low confidence: 2024IVL夏季赛季后赛.xlsx:赛后数据 (score=0)
2024IVL夏季赛季后赛.xlsx:赛后数据: ['赛后数据', '检查员用']
2024IVL秋季赛常规赛.xlsx:原始数据: ['局']
2024IVL秋季赛常规赛.xlsx:赛后数据: ['赛后数据']
⚠️  Low confidence: 2024IVL秋季赛季后赛.xlsx:赛后数据 (score=0)
2024IVL秋季赛季后赛.xlsx:赛后数据: ['45', '赛后数据', '检查员用']
2025IVL夏季赛常规赛.xlsx:原始数据: ['局']
2025IVL夏季赛常规赛.xlsx:赛后数据: ['逃脱数据错误']
2025IVL夏季赛季后赛.xlsx:赛后数据: ['逃脱数据错误']
2025IVL秋季赛常规赛.xlsx:原始数据: ['局']
2025IVL秋季赛常规赛.xlsx:赛后数据: ['逃脱数据错误']
2025IVL秋季赛季后赛.xlsx:赛后数据: ['逃脱数据错误']
⚠️  Low confidence: 2024IJL夏季赛常规赛.xlsx:赛后数据 (score=0)
2024IJL夏季赛常规赛.xlsx:赛后数据: ['赛后数据', '检查员用']
⚠️  Low confidence: 2024IJL秋季赛季后赛.xlsx:赛后数据 (score=0)
2024IJL秋季赛季后赛.xlsx:赛后数据: ['赛后数据', '检查员用']
2025IJL夏季赛常规赛.xlsx:原始数据: ['局']
2025IJL夏季赛常规赛.xlsx:赛后数据: ['逃脱数据错误']
2025IJL夏季赛季后赛.xlsx:赛后数据: ['逃脱数据错误']
2025IJL秋季赛

In [67]:
print("=== Modern file schemas ===\n")
 
# Check if all modern files have identical columns — use first file as reference
ref_file = ALL_MODERN_FILES[0]
ref_raw    = set(clean_col_names(read_modern_sheet(ref_file, MODERN_RAW_SHEET, nrows=0).columns))
ref_player = set(clean_col_names(read_modern_sheet(ref_file, MODERN_PLAYER_SHEET, nrows=0).columns))
 
print(f"Reference file: {ref_file}")
print(f"  {MODERN_RAW_SHEET} named cols ({len(ref_raw)}):    {sorted(ref_raw)}")
print(f"  {MODERN_PLAYER_SHEET} named cols ({len(ref_player)}): {sorted(ref_player)}")
print()
 
print("--- Checking all modern files for schema deviations ---\n")
deviations = {}
 
for f in ALL_MODERN_FILES[1:]:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        continue
 
    raw_cols    = set(clean_col_names(read_modern_sheet(f, MODERN_RAW_SHEET,    nrows=0).columns))
    player_cols = set(clean_col_names(read_modern_sheet(f, MODERN_PLAYER_SHEET, nrows=0).columns))
 
    raw_diff    = raw_cols    ^ ref_raw      # symmetric difference
    player_diff = player_cols ^ ref_player
 
    if raw_diff or player_diff:
        print(f"⚠️  {f}")
        if raw_diff:
            print(f"   原始数据 differs:  {raw_diff}")
        if player_diff:
            print(f"   赛后数据 differs: {player_diff}")
        deviations[f] = {'raw': raw_diff, 'player': player_diff}
    else:
        print(f"✅ {f}")
 
if not deviations:
    print("\n✅ All modern files have identical schemas")
 
 

=== Modern file schemas ===

⚠️  Low confidence: 2024IVL夏季赛常规赛.xlsx:赛后数据 (score=0)
Reference file: 2024IVL夏季赛常规赛.xlsx
  原始数据 named cols (34):    ['_header_row', '_source', 'away_score', 'away_team', 'banned_map', 'day', 'game_score', 'half', 'half_score', 'home_score', 'home_team', 'hunter_character', 'hunter_player', 'hunter_side', 'hunter_team', 'map_name', 'map_picker_side', 'map_picker_team', 'match_num', 'match_score', 'month', 'page_error', 'phase1_hunter_ban', 'phase1_survivor_ban', 'phase1_survivor_pick', 'phase2_survivor_ban', 'phase2_survivor_pick', 'phase3_survivor_ban', 'phase3_survivor_pick', 'side_picker_team', 'survivor_first_ban_auto', 'survivor_team', 'time', 'winner_side']
  赛后数据 named cols (56): ['_header_row', '_source', 'away_team', 'day', 'duration_min', 'duration_sec', 'game_duration_sec', 'gens_remaining', 'half', 'home_team', 'hunter_character', 'match_num', 'match_num_2', 'month', 'mvp', 'notes', 'page_error', 'paused_or_rematch', 'survivor1_boards', 'survivor

In [68]:
print("=== Column availability across eras ===\n")
 
cols_2020   = legacy_schemas['2020原始']
cols_2023   = legacy_schemas['2023原始']
cols_modern = ref_raw | ref_player   # union of both modern sheets
 
only_2023_plus = cols_2023 - cols_2020
only_modern    = cols_modern - cols_2023
lost_in_modern = cols_2023 - cols_modern   # cols in 2023 but gone in modern
 
print("Columns that appeared in 2023 but NOT in 2020")
print("(i.e. you can't use these for pre-2023 data):")
for c in sorted(only_2023_plus):
    print(f"  {c}")
 
print(f"\nColumns new in modern format (not in 2023 raw):")
for c in sorted(only_modern):
    print(f"  {c}")
 
print(f"\nColumns in 2023 legacy but gone in modern (check if renamed):")
for c in sorted(lost_in_modern):
    print(f"  {c}")
 
 

=== Column availability across eras ===

Columns that appeared in 2023 but NOT in 2020
(i.e. you can't use these for pre-2023 data):
  banned_map
  duration_min
  duration_sec
  half
  map_picker_team
  mvp
  paused_or_rematch
  phase1_hunter_ban
  phase1_survivor_ban
  phase1_survivor_pick
  phase2_survivor_ban
  phase2_survivor_pick
  phase3_survivor_ban
  phase3_survivor_pick
  side_picker_team
  survivor1_boards
  survivor1_heals
  survivor1_rescues
  survivor1_result
  survivor2_boards
  survivor2_heals
  survivor2_rescues
  survivor2_result
  survivor3_boards
  survivor3_heals
  survivor3_rescues
  survivor3_result
  survivor4_boards
  survivor4_heals
  survivor4_rescues
  survivor4_result
  survivor_first_ban_auto
  survivor_team
  time

Columns new in modern format (not in 2023 raw):
  day
  game_duration_sec
  match_num_2
  month
  page_error
  total_escapes
  total_harassment
  total_repair_progress
  检查员用
  赛后数据

Columns in 2023 legacy but gone in modern (check if renamed):


In [74]:
print("=== Row counts ===\n")
 
total_rows = 0
 
print("Legacy tabs:")
for tab in LEGACY_RAW_TABS + LEGACY_COA_TABS:
    df = read_legacy_tab(tab)
    # Drop completely empty rows
    df_clean = df.dropna(how='all')
    print(f"  {tab}: {len(df_clean)} rows")
    total_rows += len(df_clean)
 
print("\nModern files (原始数据):")
for f in ALL_MODERN_FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        continue
    df = read_modern_sheet(f, MODERN_RAW_SHEET)
    df_clean = df.dropna(how='all')
    print(f"  {f}: {len(df_clean)} rows")
    total_rows += len(df_clean)
 
print(f"\nTotal rows across all included files: {total_rows}")
 
 

=== Row counts ===

Legacy tabs:
  2020原始: 1142 rows
  2021原始: 1117 rows
  2022原始: 1117 rows
  2023原始: 1160 rows
  COA4: 90 rows
  COA5: 208 rows
  COA6: 294 rows

Modern files (原始数据):
  2024IVL夏季赛常规赛.xlsx: 496 rows
  2024IVL夏季赛季后赛.xlsx: 74 rows
  2024IVL秋季赛常规赛.xlsx: 508 rows
  2024IVL秋季赛季后赛.xlsx: 71 rows
  2025IVL夏季赛常规赛.xlsx: 502 rows
  2025IVL夏季赛季后赛.xlsx: 96 rows
  2025IVL秋季赛常规赛.xlsx: 486 rows
  2025IVL秋季赛季后赛.xlsx: 108 rows
  2024IJL夏季赛常规赛.xlsx: 230 rows
  2024IJL秋季赛季后赛.xlsx: 57 rows
  2025IJL夏季赛常规赛.xlsx: 312 rows
  2025IJL夏季赛季后赛.xlsx: 70 rows
  2025IJL秋季赛常规赛.xlsx: 304 rows
  2025IJL秋季赛季后赛.xlsx: 72 rows
  COA8 全球总决赛小组赛.xlsx: 206 rows
  COA8 全球总决赛淘汰赛.xlsx: 72 rows
  COA9 全球总决赛小组赛.xlsx: 218 rows
  COA9 全球总决赛淘汰赛.xlsx: 92 rows

Total rows across all included files: 9102


In [53]:
print("=== Missing values in key columns (modern 原始数据) ===\n")
 
KEY_COLS_RAW = [
    '主场', '客场', '场次', '半场',
    '屠队主客', '人队', '屠队', '屠名',
    '胜利方', '地图', '屠选', '剩机台数'
]
 
KEY_COLS_PLAYER = [
    '人ID1', '人ID2', '人ID3', '人ID4',
    '角色', '修机', '救人', '牵制', '结果',
    '屠ID', '角色.4', '剩机', '命中', '击倒'
]
 
# Aggregate across all modern files
all_raw = []
all_player = []
 
for f in ALL_MODERN_FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        continue
    all_raw.append(read_modern_sheet(f, MODERN_RAW_SHEET))
    all_player.append(read_modern_sheet(f, MODERN_PLAYER_SHEET))
 
combined_raw    = pd.concat(all_raw,    ignore_index=True).dropna(how='all')
combined_player = pd.concat(all_player, ignore_index=True).dropna(how='all')
 
print(f"Combined modern 原始数据: {len(combined_raw)} rows")
print(f"Combined modern 赛后数据: {len(combined_player)} rows\n")
 
print("Missing in 原始数据 key columns:")
for col in KEY_COLS_RAW:
    if col in combined_raw.columns:
        n = combined_raw[col].isnull().sum()
        pct = n / len(combined_raw) * 100
        flag = "⚠️ " if pct > 5 else "  "
        print(f"  {flag}{col}: {n} missing ({pct:.1f}%)")
    else:
        print(f"  ❓ {col}: NOT FOUND in columns")
 
print("\nMissing in 赛后数据 key columns:")
for col in KEY_COLS_PLAYER:
    if col in combined_player.columns:
        n = combined_player[col].isnull().sum()
        pct = n / len(combined_player) * 100
        flag = "⚠️ " if pct > 5 else "  "
        print(f"  {flag}{col}: {n} missing ({pct:.1f}%)")
    else:
        print(f"  ❓ {col}: NOT FOUND in columns")
 
 

=== Missing values in key columns (modern 原始数据) ===

⚠️  Low confidence: 2024IVL夏季赛常规赛.xlsx:赛后数据 (score=0)
⚠️  Low confidence: 2024IVL夏季赛季后赛.xlsx:赛后数据 (score=0)
⚠️  Low confidence: 2024IVL秋季赛季后赛.xlsx:赛后数据 (score=0)
⚠️  Low confidence: 2024IJL夏季赛常规赛.xlsx:赛后数据 (score=0)
⚠️  Low confidence: 2024IJL秋季赛季后赛.xlsx:赛后数据 (score=0)


InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [13]:
print("=== Player ID audit ===\n")
 
import difflib
 
# Collect all player IDs from modern files
hunter_ids   = []
survivor_ids = []
 
for f in ALL_MODERN_FILES:
    path = os.path.join(DATA_DIR, f)
    if not os.path.exists(path):
        continue
    df_p = read_modern_sheet(f, MODERN_PLAYER_SHEET).dropna(how='all')
 
    if '屠ID' in df_p.columns:
        hunter_ids.extend(df_p['屠ID'].dropna().astype(str).str.strip().tolist())
 
    for col in ['人ID1', '人ID2', '人ID3', '人ID4']:
        if col in df_p.columns:
            survivor_ids.extend(df_p[col].dropna().astype(str).str.strip().tolist())
 
unique_hunters   = sorted(set(h.lower() for h in hunter_ids   if h not in ('nan', '')))
unique_survivors = sorted(set(s.lower() for s in survivor_ids if s not in ('nan', '')))
all_players      = sorted(set(unique_hunters) | set(unique_survivors))
 
print(f"Unique hunter IDs:   {len(unique_hunters)}")
print(f"Unique survivor IDs: {len(unique_survivors)}")
print(f"Unique players total (union): {len(all_players)}")
 
# Check for probable duplicate IDs (typos, capitalisation changes)
print("\n--- Probable duplicate IDs (similarity > 0.85) ---")
duplicates_found = False
checked = set()
for name in all_players:
    if name in checked:
        continue
    candidates = [p for p in all_players if p != name and p not in checked]
    matches = difflib.get_close_matches(name, candidates, n=3, cutoff=0.85)
    if matches:
        print(f"  '{name}'  ~  {matches}")
        duplicates_found = True
    checked.add(name)
 
if not duplicates_found:
    print("  None found ✅")
 
# Check ppxia specifically (known dual-role player)
ppxia_variants = [p for p in all_players if 'ppx' in p.lower()]
print(f"\nppxia variants found: {ppxia_variants}")
 
# Check for other dual-role players (appear in both hunter and survivor lists)
dual_role = set(unique_hunters) & set(unique_survivors)
print(f"\nPlayers appearing as both hunter and survivor: {len(dual_role)}")
for p in sorted(dual_role):
    h_count = hunter_ids.count(p) + hunter_ids.count(p.upper())
    s_count = sum(survivor_ids.count(x) for x in [p, p.upper()])
    print(f"  {p}: ~{hunter_ids.count(p)} hunter games, ~{s_count} survivor games")
 
 

=== Player ID audit ===

Unique hunter IDs:   62
Unique survivor IDs: 170
Unique players total (union): 231

--- Probable duplicate IDs (similarity > 0.85) ---
  'lin'  ~  ['lion']
  'lyan'  ~  ['yan']
  'nyan'  ~  ['yan']
  'pipicha'  ~  ['ppicha']
  'stian'  ~  ['tian']
  'yan'  ~  ['zyan', 'yuan']
  'yuan'  ~  ['zyuan']
  'zyan'  ~  ['zyuan']

ppxia variants found: ['ppxia']

Players appearing as both hunter and survivor: 1
  brontal: ~55 hunter games, ~4 survivor games


In [ ]:
print("=== Outcome distributions ===\n")
 
print("胜利方 (winner) value counts — modern 原始数据:")
print(combined_raw['胜利方'].value_counts(dropna=False))
 
# Hunter win rate
winner_col = '胜利方'
hunter_side_col = '屠队主客'
 
if winner_col in combined_raw.columns and hunter_side_col in combined_raw.columns:
    df_outcomes = combined_raw[[winner_col, hunter_side_col, '人队', '屠队']].dropna(
        subset=[winner_col]
    )
 
    # Hunter wins when 胜利方 == 屠 (hunter side wins)
    hunter_wins = (df_outcomes[winner_col] == '屠').sum()
    survivor_wins = (df_outcomes[winner_col] == '人').sum()
    draws = len(df_outcomes) - hunter_wins - survivor_wins
 
    total = len(df_outcomes)
    print(f"\nHunter wins:   {hunter_wins} ({hunter_wins/total*100:.1f}%)")
    print(f"Survivor wins: {survivor_wins} ({survivor_wins/total*100:.1f}%)")
    print(f"Draws/other:   {draws} ({draws/total*100:.1f}%)")
    print(f"Total games:   {total}")
 
# Remaining generators distribution
gen_col = '剩机台数'
if gen_col in combined_raw.columns:
    print(f"\nRemaining generators distribution:")
    print(combined_raw[gen_col].value_counts().sort_index())
    impossible = combined_raw[(combined_raw[gen_col] < 0) |
                               (combined_raw[gen_col] > 5)]
    print(f"\nRows with impossible generator values: {len(impossible)}")
 
 

In [15]:
print("=== Map coverage ===\n")
 
map_col = '地图'
if map_col in combined_raw.columns:
    map_counts = combined_raw[map_col].value_counts()
    print(map_counts.to_string())
    low_maps = map_counts[map_counts < 20]
    if len(low_maps):
        print(f"\n⚠️  Maps with <20 appearances (unreliable fixed effects):")
        print(low_maps.to_string())
 
print("\n=== Hunter character coverage ===\n")
hunter_char_col = '屠选'
if hunter_char_col in combined_raw.columns:
    char_counts = combined_raw[hunter_char_col].value_counts()
    print(char_counts.to_string())
    low_chars = char_counts[char_counts < 20]
    if len(low_chars):
        print(f"\n⚠️  Characters with <20 appearances (unreliable baselines):")
        print(low_chars.to_string())
 

=== Map coverage ===

地图
湖景村      449
永眠镇      442
月亮河公园    410
红教堂      268
军工厂      244
唐人街      238
里奥的回忆    212
不归林      149
圣心医院      86

=== Hunter character coverage ===

屠选
跛脚羊      506
喧嚣       413
歌剧演员     396
时空之影     235
杂货商      167
26号守卫    140
鹿头        84
蜡像师       78
使徒        76
梦之女巫      59
台球手       54
破轮        53
小丑        53
蜘蛛        39
隐士        29
记录员       27
噩梦        24
宿伞之魂      20
守夜人       14
红蝶        13
雕刻家       11
红夫人       10
渔女         9
爱哭鬼        5
孽蜥         4
博士         3
厂长         1

⚠️  Characters with <20 appearances (unreliable baselines):
屠选
守夜人    14
红蝶     13
雕刻家    11
红夫人    10
渔女      9
爱哭鬼     5
孽蜥      4
博士      3
厂长      1


In [55]:
print("=== Missing values in legacy raw tabs ===\n")
 
LEGACY_KEY_COLS = {
    '2020原始': ['主场', '客场', '屠队', '屠名', '胜利方', '地图', '屠选'],
    '2021原始': ['主场', '客场', '人队', '屠队', '屠名', '胜利方', '地图', '屠选', '结果'],
    '2022原始': ['主场', '客场', '人队', '屠队', '屠名', '胜利方', '地图', '屠选', '结果'],
    '2023原始': ['主场', '客场', '人队', '屠队', '屠名', '胜利方', '地图', '屠选', '结果', '砸板命中'],
}
 
for tab, key_cols in LEGACY_KEY_COLS.items():
    df = read_legacy_tab(tab).dropna(how='all')
    print(f"{tab} ({len(df)} rows):")
    for col in key_cols:
        if col in df.columns:
            n = df[col].isnull().sum()
            pct = n / len(df) * 100
            flag = "⚠️ " if pct > 5 else "  "
            print(f"  {flag}{col}: {n} missing ({pct:.1f}%)")
        else:
            print(f"  ❓ {col}: NOT IN THIS TAB")
    print()
 

=== Missing values in legacy raw tabs ===

2020原始 (1632 rows):
  ❓ 主场: NOT IN THIS TAB
  ❓ 客场: NOT IN THIS TAB
  ❓ 屠队: NOT IN THIS TAB
  ❓ 屠名: NOT IN THIS TAB
  ❓ 胜利方: NOT IN THIS TAB
  ❓ 地图: NOT IN THIS TAB
  ❓ 屠选: NOT IN THIS TAB

2021原始 (1632 rows):
  ❓ 主场: NOT IN THIS TAB
  ❓ 客场: NOT IN THIS TAB
  ❓ 人队: NOT IN THIS TAB
  ❓ 屠队: NOT IN THIS TAB
  ❓ 屠名: NOT IN THIS TAB
  ❓ 胜利方: NOT IN THIS TAB
  ❓ 地图: NOT IN THIS TAB
  ❓ 屠选: NOT IN THIS TAB
  ❓ 结果: NOT IN THIS TAB

2022原始 (1632 rows):
  ❓ 主场: NOT IN THIS TAB
  ❓ 客场: NOT IN THIS TAB
  ❓ 人队: NOT IN THIS TAB
  ❓ 屠队: NOT IN THIS TAB
  ❓ 屠名: NOT IN THIS TAB
  ❓ 胜利方: NOT IN THIS TAB
  ❓ 地图: NOT IN THIS TAB
  ❓ 屠选: NOT IN THIS TAB
  ❓ 结果: NOT IN THIS TAB

2023原始 (1632 rows):
  ❓ 主场: NOT IN THIS TAB
  ❓ 客场: NOT IN THIS TAB
  ❓ 人队: NOT IN THIS TAB
  ❓ 屠队: NOT IN THIS TAB
  ❓ 屠名: NOT IN THIS TAB
  ❓ 胜利方: NOT IN THIS TAB
  ❓ 地图: NOT IN THIS TAB
  ❓ 屠选: NOT IN THIS TAB
  ❓ 结果: NOT IN THIS TAB
  ❓ 砸板命中: NOT IN THIS TAB



In [17]:
print("""
=== AUDIT SUMMARY — fill in after reviewing output above ===
 
FILE COVERAGE:
  [ ] All expected files present
  [ ] COA7 confirmed missing — noted in README
  [ ] COA8 Japan qualifier renamed correctly
 
SCHEMA FINDINGS:
  2020: missing 半场 (game half), 人队 (survivor team), 砸板 (board breaks),
        救人 (rescues), 治疗 (heals), 结果 (outcome per player)
  2021+: game halves and rescue data added
  2023+: board breaks and heals added
  Modern: player stats moved to separate 赛后数据 sheet
 
MODELING IMPLICATIONS:
  Bradley-Terry (team/player skill):  can use ALL years
  Mixed effects (variance decomp):    can use ALL years (2021+ for halves)
  Player efficiency metric:           use 2023+ only (needs board breaks + heals)
 
PLAYER ID ISSUES FOUND:
  [ ] List any duplicate IDs here after running Cell 10
  [ ] ppxia dual-role confirmed
  [ ] Other dual-role players: ___
 
OUTCOME DISTRIBUTION:
  [ ] Hunter win rate: ____%
  [ ] Any suspicious draws or impossible values: ___
 
LOW COVERAGE WARNINGS:
  [ ] Maps with <20 games: ___
  [ ] Characters with <20 games: ___
 
NEXT STEP: build column mapper in src/ingest.py
""")
 


=== AUDIT SUMMARY — fill in after reviewing output above ===

FILE COVERAGE:
  [ ] All expected files present
  [ ] COA7 confirmed missing — noted in README
  [ ] COA8 Japan qualifier renamed correctly

SCHEMA FINDINGS:
  2020: missing 半场 (game half), 人队 (survivor team), 砸板 (board breaks),
        救人 (rescues), 治疗 (heals), 结果 (outcome per player)
  2021+: game halves and rescue data added
  2023+: board breaks and heals added
  Modern: player stats moved to separate 赛后数据 sheet

MODELING IMPLICATIONS:
  Bradley-Terry (team/player skill):  can use ALL years
  Mixed effects (variance decomp):    can use ALL years (2021+ for halves)
  Player efficiency metric:           use 2023+ only (needs board breaks + heals)

PLAYER ID ISSUES FOUND:
  [ ] List any duplicate IDs here after running Cell 10
  [ ] ppxia dual-role confirmed
  [ ] Other dual-role players: ___

OUTCOME DISTRIBUTION:
  [ ] Hunter win rate: ____%
  [ ] Any suspicious draws or impossible values: ___

LOW COVERAGE WARNINGS:
  [

In [18]:
test_file = '2024IVL夏季赛常规赛.xlsx'
path = os.path.join(DATA_DIR, test_file)

for h in [0, 1, 2, 3]:
    df = pd.read_excel(path, sheet_name='原始数据', header=h, nrows=2)
    cols = [c for c in df.columns if not str(c).startswith('Unnamed')][:10]
    print(f"\nheader={h} first 10 named cols:")
    print(f"  {cols}")


header=0 first 10 named cols:
  ['本页出错', '月', '日', '时间', '主场', '客场', '场次', '半场', '屠队主客', '人队']

header=1 first 10 named cols:
  [0, 6, 8, 1400, 'TE', 'GG', 1, '上', '主', 'GG.1']

header=2 first 10 named cols:
  [0, 6, 8, 1400, 'TE', 'GG', 1, '下', '客', 'TE.1']

header=3 first 10 named cols:
  [0, 6, 8, 1400, 'TE', 'GG', 2, '上', '客', 'TE.1']
